# Pra-Pemrosesan Data (*Data Preprocessing*) & Ekstraksi Fitur Deret Waktu Polutan Udara

Data konsentrasi polutan udara harian yang diperoleh dari observasi satelit (**Sentinel-5P**) pada hakikatnya merupakan data deret waktu mentah (*raw time series*). Dalam kondisi riil, data satelit ini sering kali memiliki kendala kualitas data:
1. **Nilai Hilang (*Missing Values*):** Disebabkan oleh tutupan awan tebal (*cloud masking*), anomali sensor, atau kualitas piksel satelit yang berada di bawah ambang batas validitas (`qa_value < 0.5`).
2. **Pencilan Ekstrem (*Outliers*):** Lonjakan nilai konsentrasi yang tidak wajar akibat pantulan aerosol permukaan, anomali pantulan sinar matahari (*glint*), atau gangguan instrumen pengukur.
3. **Format Tidak Standar / Nilai Kosong:** Keberadaan string non-numerik yang dapat memicu *crash* saat komputasi numerik.

Jika data mentah yang bermasalah langsung diumpankan ke algoritma ekstraksi fitur deret waktu (**TSFEL**), hasilnya akan sangat bias (*Garbage In, Garbage Out*), bahkan dapat memicu lonjakan nilai fitur hingga skala $10^8$ yang merusak analisis lanjutan seperti PCA dan K-Means Clustering.

Oleh karena itu, notebook ini membagi proses pra-pemrosesan data ke dalam **7 tahapan terstruktur**:

```
[1. Pemuatan & Kronologi Datetime] ──► [2. Sanitasi Tipe Data Numerik] ──► [3. Filter Outlier (IQR)]
                                                                                  │ (ubah ke NaN)
                                                                                  ▼
[6. Ekstraksi 68 Fitur TSFEL] ◄── [5. Pembentukan Sinyal 1D (fs=1)] ◄── [4. Interpolasi Waktu]
             │
             ▼
[7. Ekspor CSV Fitur Siap Klastering]
```

---

### Ringkasan 7 Tahapan Pra-Pemrosesan:
| No | Tahapan | Fungsi / Metode Utama | Tujuan & Manfaat |
|:---|:---|:---|:---|
| **1** | Pemuatan & Kronologi Waktu | `pd.to_datetime()`, `sort_values('date')` | Menjamin kontinuitas urutan kalender observasi harian. |
| **2** | Sanitasi Numerik | `pd.to_numeric(..., errors='coerce')` | Mengubah string/karakter kotor menjadi `NaN` secara aman. |
| **3** | Deteksi Pencilan (Outlier) | Batas Pagar IQR ($Q_1 - 1.5 \times \text{IQR}$, $Q_3 + 1.5 \times \text{IQR}$) | Mengisolasi anomali ekstrem menjadi `NaN` tanpa memotong baris waktu. |
| **4** | Imputasi Data Hilang | `.interpolate(method='time').ffill().bfill()` | Mengisi kekosongan data dengan memperhitungkan jarak hari sesungguhnya. |
| **5** | Persiapan Sinyal 1D | `.astype(float).values`, `fs = 1` | Membentuk larik NumPy 1-dimensi dengan frekuensi sampling 1 hari. |
| **6** | Ekstraksi Fitur TSFEL | `FEATURE_LIST` (68 fitur), `inspect.signature` | Mengekstrak representasi domain temporal, statistik, dan spektral secara dinamis. |
| **7** | Ekspor Berkas Fitur | `.to_csv(..., index=False)` | Menyimpan dataset fitur yang siap digabungkan untuk PCA & K-Means. |


---
# BAGIAN I: STEP-BY-STEP PRA-PEMROSESAN & EKSTRAKSI FITUR (STUDI KASUS NO2)

Pada bagian ini, kita akan membedah setiap langkah secara bertahap pada variabel polutan **NO2 (Nitrogen Dioksida)** di wilayah Sambeng, Lamongan.

---
### Langkah 1: Pemuatan Data & Pengurutan Kronologis Waktu (*Time Series Setup*)

**Mengapa ini penting?**
- Data deret waktu (*time series*) sangat bergantung pada urutan kronologis. Jika data tidak terurut berdasarkan tanggal, algoritma autokorelasi, tren, dan frekuensi spektral akan salah membaca urutan sinyal.
- Fungsi `pd.to_datetime()` mengonversi string tanggal (misal `'2025-08-25'`) menjadi objek representasi waktu standar `Timestamp`.
- Fungsi `sort_values('date').reset_index(drop=True)` memastikan data tersusun rapi dari hari pertama hingga hari terakhir observasi.


In [ ]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# 1. Pemuatan dataset mentah deret waktu polutan Sambeng
file_path = 'all_polutan_Sambeng_timeseries.csv'
df_raw = pd.read_csv(file_path)

# 2. Salin data dan konversi kolom tanggal menjadi objek Datetime
df = df_raw.copy()
df['date'] = pd.to_datetime(df['date'])

# 3. Urutkan berdasarkan kronologi waktu menaik (ascending)
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

print(f"Dataset berhasil dimuat!")
print(f"- Jumlah baris observasi : {df.shape[0]} hari")
print(f"- Rentang tanggal        : {df['date'].min().strftime('%Y-%m-%d')} s.d. {df['date'].max().strftime('%Y-%m-%d')}")
print(f"- Variabel target        : {target_pollutant}")
print("\nLima baris pertama data:")
print(df[['date', target_pollutant]].head())


---
### Langkah 2: Sanitasi Tipe Data Numerik (*Type Casting & Coercion*)

**Permasalahan:**
Sering kali data mentah hasil crawling satelit mengandung nilai kosong bertipe teks (`"None"`, `"null"`, `"N/A"`, atau spasi kosong). Jika langsung diproses secara matematis, Python akan melempar error `TypeError: could not convert string to float`.

**Solusi:**
Fungsi `pd.to_numeric(df[target_pollutant], errors='coerce')`:
- Parameter `errors='coerce'` akan memaksa seluruh karakter yang tidak bisa dikonversi menjadi angka desimal untuk diubah menjadi `NaN` (*Not a Number*).
- Dengan cara ini, proses komputasi tidak akan terhenti (*crash*), dan kita dapat menghitung persis berapa banyak nilai yang hilang (*missing values*) pada data observasi.


In [ ]:
# Paksa kolom target menjadi tipe data numerik (float)
# Karakter string kotor otomatis dikonversi menjadi NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

# Hitung jumlah nilai hilang sebelum proses pembersihan
n_missing_before = df[target_pollutant].isna().sum()
total_rows = len(df)
pct_missing = (n_missing_before / total_rows) * 100

print(f"Hasil Sanitasi Tipe Data Numerik:")
print(f"- Total baris observasi          : {total_rows}")
print(f"- Nilai NaN / hilang awal        : {n_missing_before} baris ({pct_missing:.2f}%)")
print(f"- Nilai valid awal               : {total_rows - n_missing_before} baris")


---
### Langkah 3: Deteksi & Penanganan Pencilan (*Outliers*) dengan Metode IQR

**Konsep Statistik IQR (*Interquartile Range*):**
Pencilan atau nilai ekstrem dapat merusak komputasi statistik (seperti rata-rata, variansi, skewness, kurtosis, dan energi spektral). Untuk mendeteksinya, kita menggunakan metode non-parametrik yang tangguh terhadap distribusi data miring:

1. **Kuartil 1 ($Q_1$):** Titik persentil ke-25 dari data.
2. **Kuartil 3 ($Q_3$):** Titik persentil ke-75 dari data.
3. **Rentang Interkuartil (IQR):**
   $$\text{IQR} = Q_3 - Q_1$$
4. **Batas Bawah Pagar (*Lower Bound*):**
   $$\text{Batas Bawah} = Q_1 - 1.5 \times \text{IQR}$$
5. **Batas Atas Pagar (*Upper Bound*):**
   $$\text{Batas Atas} = Q_3 + 1.5 \times \text{IQR}$$

**Mengapa Outlier Diubah Menjadi `NaN` dan BUKAN Dihapus Barisnya?**
> **Prinsip Penting Deret Waktu:**
> Jika kita menghapus baris data yang mengandung pencilan (`df.dropna()`), urutan kalender harian akan terputus (misalnya: tanggal 10 langsung lompat ke tanggal 14).
> Hal ini merusak sifat periodisitas deret waktu (*time series continuity*) dan menyebabkan frekuensi sampling menjadi tidak beraturan.
> Oleh karena itu, nilai di luar rentang $[ \text{Batas Bawah}, \text{Batas Atas} ]$ **diubah menjadi `NaN`** untuk kemudian ditambal secara kontinu pada tahap interpolasi.


In [ ]:
# 1. Hitung Kuartil 1, Kuartil 3, dan IQR
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1

# 2. Tentukan Batas Bawah dan Batas Atas toleransi
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Hitung berapa banyak titik outlier yang terdeteksi
outlier_mask = (df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound)
n_outliers = outlier_mask.sum()

print("Analisis Statistik Pencilan (IQR):")
print(f"- Q1 (Persentil 25)   : {Q1:.8e}")
print(f"- Q3 (Persentil 75)   : {Q3:.8e}")
print(f"- Nilai IQR           : {IQR:.8e}")
print(f"- Batas Bawah Pagar   : {lower_bound:.8e}")
print(f"- Batas Atas Pagar    : {upper_bound:.8e}")
print(f"- Jumlah Outlier      : {n_outliers} titik terdeteksi")

# 4. Ganti nilai outlier menjadi NaN (tanpa membuang baris kalender)
df.loc[outlier_mask, target_pollutant] = np.nan

total_nan_now = df[target_pollutant].isna().sum()
print(f"\nTotal nilai NaN saat ini (Missing Awal + Outlier): {total_nan_now} baris")


---
### Langkah 4: Imputasi Nilai Hilang Menggunakan Interpolasi Waktu (*Time-based Interpolation*)

Setelah nilai hilang awal dan titik-titik pencilan disatukan menjadi nilai `NaN`, langkah berikutnya adalah mengimputasi kekosongan tersebut agar deret waktu kembali utuh dan kontinu.

Sintaks yang digunakan:
$$\text{df\_clean} = \text{df.set\_index('date').interpolate(method='time').ffill().bfill()}$$

**Bedah Komponen Kode:**
1. **`set_index('date')`:**
   Menjadikan kolom tanggal sebagai indeks DataFrame. Hal ini merupakan syarat wajib untuk menjalankan interpolasi berbasis waktu di pustaka Pandas.
2. **`interpolate(method='time')`:**
   - Berbeda dengan interpolasi linier biasa (`method='linear'`) yang memperlakukan jarak antar-baris selalu sama (1 unit), `method='time'` memperhitungkan selisih durasi tanggal yang sesungguhnya (misal jika ada selisih 1 hari vs 3 hari).
   - Nilai estimasi dihitung secara linier proporsional terhadap jarak waktu antara titik valid sebelum dan sesudah kekosongan.
3. **`.ffill()` (*Forward Fill*) dan `.bfill()` (*Backward Fill*):**
   - **Kelemahan Inheren Interpolasi Linier:** Interpolasi hanya dapat bekerja jika titik `NaN` diapit oleh data valid di sebelah kiri dan kanan. Jika nilai `NaN` berada di ujung awal data (hari ke-1 belum ada data valid) atau di ujung akhir data, interpolasi tidak dapat memprediksi nilainya (*ekstrapolasi*).
   - `.bfill()` mengisi kekosongan di ujung awal dengan menarik nilai valid terdekat di depannya.
   - `.ffill()` mengisi kekosongan di ujung akhir dengan meneruskan nilai valid terakhir yang diketahui.
   - Kombinasi keduanya menjamin bahwa deret waktu menjadi **100% lengkap tanpa ada satu pun nilai `NaN` yang tersisa (*gapless time series*)**.


In [ ]:
# Imputasi data hilang dengan interpolasi berbasis waktu + forward/backward fill
df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

# Verifikasi bahwa tidak ada lagi nilai NaN yang tersisa
n_missing_after = df_clean[target_pollutant].isna().sum()

print("Hasil Imputasi Interpolasi Waktu:")
print(f"- Nilai NaN sebelum interpolasi : {total_nan_now}")
print(f"- Nilai NaN setelah interpolasi : {n_missing_after} (100% Lengkap & Kontinu)")
print("\nContoh 5 baris pertama data bersih setelah interpolasi:")
print(df_clean[[target_pollutant]].head())


---
### Langkah 5: Pembentukan Sinyal 1-Dimensi & Frekuensi Sampling (`fs = 1`)

**Persyaratan Pustaka TSFEL:**
Pustaka *Time Series Feature Extraction Library* (TSFEL) dirancang untuk memproses sinyal deret waktu dalam bentuk larik 1-dimensi (*NumPy 1D array*) dengan elemen bertipe numerik desimal (*float*).

**Arti Parameter Frekuensi Sampling (`fs = 1`):**
- Frekuensi sampling ($f_s$) menyatakan banyaknya sampel pengamatan per satuan waktu.
- Karena data konsentrasi polutan satelit ini diobservasi dengan resolusi **harian (1 observasi per hari)**, maka kita tetapkan:
  $$f_s = 1 \text{ sampel/hari}$$
- Nilai $f_s$ ini menjadi referensi fundamental untuk komputasi fitur berbasis frekuensi dan domain waktu, seperti *fundamental frequency*, *spectral centroid*, *power bandwidth*, dan *spectral roll-off*.


In [ ]:
# Tetapkan frekuensi sampling (1 sampel per hari)
fs = 1

# Konversi kolom polutan bersih menjadi array 1D NumPy berjenis float
signal_1d = df_clean[target_pollutant].astype(float).values

print("Spesifikasi Sinyal Deret Waktu:")
print(f"- Tipe data array    : {type(signal_1d)}")
print(f"- Tipe elemen data   : {signal_1d.dtype}")
print(f"- Dimensi sinyal     : {signal_1d.shape} (Panjang {len(signal_1d)} hari)")
print(f"- Frekuensi sampling : fs = {fs} Hz/hari")
print(f"- Nilai minimum      : {signal_1d.min():.8e}")
print(f"- Nilai maksimum     : {signal_1d.max():.8e}")
print(f"- Nilai rata-rata    : {signal_1d.mean():.8e}")


---
### Langkah 6: Ekstraksi 68 Fitur TSFEL Dinamis & Robust

**Mengapa Diperlukan Mekanisme Ekstraksi Khusus?**
Secara *default*, modul fitur TSFEL memiliki karakteristik teknis sebagai berikut:
1. **Perbedaan Tanda Tangan Parameter Fungsi:**
   - Sebagian fungsi (seperti `calc_mean`, `calc_std`, `skewness`) hanya menerima masukan berupa sinyal `fn(signal)`.
   - Sebagian fungsi lainnya (seperti `spectral_centroid`, `fundamental_frequency`, `power_bandwidth`) mewajibkan masukan frekuensi sampling `fn(signal, fs)`.
   - Jika kita mengirim `fs` ke fungsi yang tidak membutuhkannya, Python akan melempar `TypeError: fn() got an unexpected keyword argument 'fs'`.
   - **Solusi:** Kita memanfaatkan modul bawaan Python `inspect.signature(fn).parameters` untuk memeriksa apakah fungsi tersebut membutuhkan parameter `fs` secara otomatis sebelum dipanggil.

2. **Perbedaan Format Nilai Keluaran (*Scalar Standardization*):**
   - Sebagian fungsi TSFEL mengembalikan nilai skalar tunggal (`float`), namun fungsi lain (seperti `wavelet`, `mfcc`, atau `ecdf`) dapat mengembalikan kamus (*dictionary* `{"values": ...}`) atau larik matriks (*array/list*).
   - **Solusi:** Fungsi pembantu `to_scalar(result)` mengonversi setiap keluaran multi-nilai menjadi nilai skalar rerata representatif (`np.nanmean`) agar siap disusun rapi ke dalam tabel baris-kolom (DataFrame 1 x 68).

**Cakupan 68 Fitur TSFEL:**
Fitur-fitur ini mencakup 3 domain utama deret waktu:
1. **Domain Statistik:** Mean, Median, Min, Max, Variansi, Standar Deviasi, Skewness, Kurtosis, IQR, Mean Absolute Deviation.
2. **Domain Temporal:** Abs Energy, Peak-to-Peak Distance, Zero Crossing Rate, Positive/Negative Turning Points, Petrosian/Higuchi Fractal Dimension, Lempel-Ziv Complexity, Hurst Exponent.
3. **Domain Spektral (Frekuensi):** Spectral Centroid, Spectral Spread, Spectral Decrease, Spectral Entropy, Spectral Roll-off, Fundamental Frequency, Power Bandwidth, Wavelet Energy/Entropy.


In [ ]:
# Daftar 68 fitur TSFEL yang diekstrak
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Total fitur TSFEL yang akan diekstrak: {len(FEATURE_LIST)} fitur")


def to_scalar(result):
    """
    Mengonversi berbagai tipe keluaran TSFEL (skalar, dict, list, numpy array)
    menjadi nilai float skalar tunggal yang terstandar.
    """
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    """
    Mengekstrak satu fitur TSFEL dengan inspeksi parameter dinamis (inspect.signature).
    Jika fungsi membutuhkan 'fs', kirimkan (signal, fs). Jika tidak, kirimkan (signal).
    """
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


# Ekstraksi seluruh 68 fitur untuk sinyal NO2
row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"\nProses Ekstraksi Selesai!")
print(f"Dimensi matriks fitur {target_pollutant}: {extracted_features_final.shape} (1 baris x {extracted_features_final.shape[1]} kolom fitur)")
print("\nContoh 6 fitur pertama:")
print(extracted_features_final.iloc[:, :6])


---
### Langkah 7: Ekspor Hasil Ekstraksi Fitur ke Format CSV

Matriks fitur berukuran $1 \times 68$ yang telah diekstrak kemudian diekspor ke dalam format CSV (`NO2_Sambeng_TSFEL.csv`).
Berkas CSV ini merepresentasikan karakteristik dinamis deret waktu polutan NO2 wilayah Sambeng selama 1 tahun, yang nantinya akan digabungkan bersama data dari wilayah-wilayah lain di Jawa Timur untuk analisis Reduksi Dimensi (PCA) dan K-Means Clustering.


In [ ]:
# Simpan hasil ekstraksi fitur ke berkas CSV
output_csv_no2 = f"{target_pollutant}_Sambeng_TSFEL.csv"
extracted_features_final.to_csv(output_csv_no2, index=False)

# Simpan juga dengan nama target Kerek untuk kompatibilitas kode turunan
extracted_features_final.to_csv(f"{target_pollutant}_Kerek_TSFEL.csv", index=False)

print(f"Berkas berhasil disimpan ke: '{output_csv_no2}' dan '{target_pollutant}_Kerek_TSFEL.csv'")


---
# BAGIAN II: IMPLEMENTASI MODULAR UNTUK POLUTAN CO DAN SO2

Sesuai prinsip *Clean Code* dan *Separation of Concerns* (Modular Architecture), kita membungkus seluruh alur 7 tahapan pra-pemrosesan dan ekstraksi fitur di atas ke dalam satu fungsi terisolasi:
`preprocess_and_extract_tsfel(df_input, pollutant_name, output_csv_prefix)`.

Dengan fungsi ini:
1. Pemrosesan polutan lain (seperti **CO** dan **SO2**) dapat dijalankan secara konsisten tanpa pengulangan kode (*Don't Repeat Yourself - DRY*).
2. Setiap polutan memiliki isolasi variabel tersendiri sehingga meminimalisir risiko efek samping (*side-effects*).


In [ ]:
def preprocess_and_extract_tsfel(df_input, pollutant_name, output_csv_prefix="Sambeng"):
    """
    Fungsi modular untuk menjalankan pipeline 7-tahap pra-pemrosesan data
    dan ekstraksi 68 fitur TSFEL pada variabel polutan yang dipilih.
    """
    print(f"\n{'='*60}")
    print(f"MEMPROSES VARIABEL POLUTAN: {pollutant_name}")
    print(f"{'='*60}")
    
    # 1 & 2. Konversi datetime dan paksa tipe data numerik
    df_p = df_input.copy()
    df_p['date'] = pd.to_datetime(df_p['date'])
    df_p = df_p.sort_values('date').reset_index(drop=True)
    df_p[pollutant_name] = pd.to_numeric(df_p[pollutant_name], errors='coerce')
    
    n_missing = df_p[pollutant_name].isna().sum()
    print(f"1. Missing values awal           : {n_missing} baris")
    
    # 3. Deteksi pencilan IQR & ganti menjadi NaN
    q1 = df_p[pollutant_name].quantile(0.25)
    q3 = df_p[pollutant_name].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outlier_mask = (df_p[pollutant_name] < lower) | (df_p[pollutant_name] > upper)
    n_outliers = outlier_mask.sum()
    print(f"2. Pencilan IQR terdeteksi       : {n_outliers} baris (diganti -> NaN)")
    df_p.loc[outlier_mask, pollutant_name] = np.nan
    
    # 4. Interpolasi berbasis waktu + ffill + bfill
    df_clean_p = df_p.set_index('date').interpolate(method='time').ffill().bfill()
    print(f"3. Sisa NaN setelah interpolasi  : {df_clean_p[pollutant_name].isna().sum()} (Gapless)")
    
    # 5. Konversi sinyal 1D
    sig_1d = df_clean_p[pollutant_name].astype(float).values
    fs_val = 1
    
    # 6. Ekstraksi 68 fitur TSFEL
    feat_row = {}
    for fn in FEATURE_LIST:
        feat_row[fn] = extract_one(fn, sig_1d, fs_val)
        
    df_feat = pd.DataFrame([feat_row])
    print(f"4. Ekstraksi fitur TSFEL selesai : {df_feat.shape[1]} fitur")
    
    # 7. Ekspor CSV
    csv_sambeng = f"{pollutant_name}_{output_csv_prefix}_TSFEL.csv"
    csv_kerek = f"{pollutant_name}_Kerek_TSFEL.csv"
    df_feat.to_csv(csv_sambeng, index=False)
    df_feat.to_csv(csv_kerek, index=False)
    print(f"5. Disimpan ke file              : '{csv_sambeng}' dan '{csv_kerek}'")
    
    return df_feat


---
### Eksekusi Polutan CO (Karbon Monoksida)

Tahapan pra-pemrosesan (sanitasi, eliminasi pencilan dengan IQR, interpolasi waktu, dan ekstraksi 68 fitur TSFEL) dijalankan untuk polutan **CO**.


In [ ]:
# Eksekusi pipeline terpadu untuk polutan CO
df_feat_co = preprocess_and_extract_tsfel(df_raw, "CO", output_csv_prefix="Sambeng")
print("\nContoh 6 fitur pertama CO:")
print(df_feat_co.iloc[:, :6])


---
### Eksekusi Polutan SO2 (Sulfur Dioksida)

Tahapan pra-pemrosesan (sanitasi, eliminasi pencilan dengan IQR, interpolasi waktu, dan ekstraksi 68 fitur TSFEL) dijalankan untuk polutan **SO2**.


In [ ]:
# Eksekusi pipeline terpadu untuk polutan SO2
df_feat_so2 = preprocess_and_extract_tsfel(df_raw, "SO2", output_csv_prefix="Sambeng")
print("\nContoh 6 fitur pertama SO2:")
print(df_feat_so2.iloc[:, :6])


---
# BAGIAN III: KESIMPULAN & EVALUASI PRA-PEMROSESAN DATA

Melalui rangkaian 7 tahapan pra-pemrosesan data deret waktu yang telah dipraktikkan pada notebook ini:

1. **Integritas Kronologis Terjaga:**
   Penggunaan `method='time'` pada proses interpolasi menjamin bahwa perhitungan nilai imputasi didasarkan pada interval waktu fisik antar-tanggal observasi satelit sesungguhnya, bukan sekadar urutan indeks numerik semata.
2. **Kekebalan dari Efek Distorsi Outlier:**
   Mengisolasi nilai pencilan ke dalam format `NaN` sebelum interpolasi terbukti ampuh mencegah terjadinya bias energi spektral (seperti kasus anomali $10^8$ yang kerap merusak grafik *Scatter Plot* dan analisis klastering).
3. **Data Utuh 100% (*Gapless Time Series*):**
   Kombinasi `interpolate(method='time')`, `.ffill()`, dan `.bfill()` berhasil menambal tutupan awan hingga menghasilkan 366 titik sampel harian yang utuh tanpa celah.
4. **Kesiapan Analisis Lanjutan (Machine Learning):**
   Seluruh berkas CSV keluaran (`NO2_Sambeng_TSFEL.csv`, `CO_Sambeng_TSFEL.csv`, `SO2_Sambeng_TSFEL.csv`) telah memiliki 68 dimensi fitur standar yang terkalibrasi dan siap diintegrasikan ke tahap **Reduksi Dimensi (PCA)** dan **K-Means Clustering**.
